In [1]:
# Math
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Predictive Modeling of Electric Vehicle Battery Degradation

## 1. Problem Formulation & Significance
As global electric vehicle (EV) adoption accelerates, accurately forecasting the Remaining Useful Life (RUL) and State of Health (SoH) of lithium-ion batteries is critical for vehicle valuation, warranty provisioning, and predictive maintenance. 

The two dominant battery chemistries: Nickel Manganese Cobalt (NMC) and Lithium Iron Phosphate (LFP), exhibit distinct degradation profiles under varying thermal and operational stresses. 

**Objective:**
This project transitions from exploratory data science into predictive machine learning. We aim to construct and evaluate multivariable models capable of predicting battery degradation. Because degradation modeling is highly susceptible to "feature dominance" (where algorithms rely entirely on obvious symptoms rather than underlying causes), we will conduct a three-part experiment:
1. **Baseline Modeling:** Predicting absolute SoH using all available telemetry and diagnostic data.
2. **Feature Ablation:** Removing direct diagnostic symptoms to force models to evaluate historical usage.
3. **Target Transformation:** Shifting the mathematical target from absolute capacity to a cycle-weighted degradation rate to uncover complex multivariable dependencies.

### 1.1 Dataset Assumptions & Constraints
**Disclaimer:** The dataset utilized in this project (`ev_battery_degradation_v1.csv`) is a synthetic dataset generated using a semi-empirical degradation model. 
* While it accurately reflects real-world physics, synthetic generation algorithms inherently hardcode strong linear correlations between primary variables (like Total Cycles) and the target variable (SoH). 
* A key focus of our machine learning methodology will be mitigating this synthetic linearity to prove our models can identify secondary, non-linear operational stresses (like fast-charging ratios and driving styles).

### 1.2 Mathematical Definition of the Baseline Target
In our initial phase, the target variable is the absolute State of Health ($SoH$), expressed as a percentage of the battery's original factory capacity. 

Let $C_{current}$ be the currently measurable capacity in kWh, and $C_{nominal}$ be the original capacity. 

$$SoH = \left( \frac{C_{current}}{C_{nominal}} \right) \times 100$$

In [6]:
data_path = 'data/ev_battery_degradation_v1.csv'
battery_data = pd.read_csv(data_path)

In [7]:
battery_data.shape

(10000, 13)

In [9]:
battery_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Vehicle_ID               10000 non-null  object 
 1   Car_Model                10000 non-null  object 
 2   Battery_Type             10000 non-null  object 
 3   Battery_Capacity_kWh     10000 non-null  float64
 4   Vehicle_Age_Months       10000 non-null  int64  
 5   Total_Charging_Cycles    10000 non-null  int64  
 6   Avg_Temperature_C        10000 non-null  float64
 7   Fast_Charge_Ratio        10000 non-null  float64
 8   Avg_Discharge_Rate_C     10000 non-null  float64
 9   Driving_Style            10000 non-null  object 
 10  Internal_Resistance_Ohm  10000 non-null  float64
 11  SoH_Percent              10000 non-null  float64
 12  Battery_Status           10000 non-null  object 
dtypes: float64(6), int64(2), object(5)
memory usage: 1015.8+ KB
